In [ ]:
import torch
import shutil
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as F
from torch.utils.data import Dataset, DataLoader
import os
import numpy as np
import cv2
import math
from PIL import Image
import matplotlib.pyplot as plt
import torchvision.transforms as T
from tqdm.auto import tqdm
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import torch.optim as optim
import json
import matplotlib.pyplot as plt
import seaborn as sns
import torch

# Kiểm tra GPU
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Đang chạy trên: {device}")

In [ ]:

# Check khi đã có model

import os
import torch
import torch.utils.data
from PIL import Image
import torchvision.transforms as T

# 1. CẤU HÌNH ĐƯỜNG DẪN DỮ LIỆU
# Đường dẫn đến thư mục chứa dữ liệu (dựa trên notebook của bạn)
ROOT_DIR = '/kaggle/input/processed-data/processed_mmrotate_data'
# Nếu bạn muốn dùng tập test riêng thì đổi đường dẫn ở đây, ví dụ: os.path.join(ROOT_DIR, 'test')
VALIDATION_DIR = os.path.join(ROOT_DIR, 'test') 
# Hàm load model Faster R-CNN pre-trained
def get_model_instance_segmentation(num_classes):
    # Load model pre-trained trên COCO
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)

    # Lấy số lượng input features của classifier head
    in_features = model.roi_heads.box_predictor.cls_score.in_features

    # Thay thế head cũ bằng head mới phù hợp với số class của bạn
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


# 2. CÁC HÀM HỖ TRỢ (Dataset, Transform, Collate)
def get_transform(train):
    transforms = []
    transforms.append(T.ToTensor())
    return T.Compose(transforms)

def collate_fn(batch):
    return tuple(zip(*batch))

class ShipDataset(torch.utils.data.Dataset):
    def __init__(self, root, transforms=None):
        self.root = root
        self.transforms = transforms
        self.imgs_dir = os.path.join(root, 'images')
        self.lbls_dir = os.path.join(root, 'labelTxt')
        # Lọc file ảnh hợp lệ
        self.imgs = sorted([f for f in os.listdir(self.imgs_dir) if f.endswith(('.jpg', '.png', '.jpeg', '.tif', '.bmp'))])

    def __getitem__(self, idx):
        img_path = os.path.join(self.imgs_dir, self.imgs[idx])
        # Convert RGB để tránh lỗi ảnh xám/RGBA
        img = Image.open(img_path).convert("RGB")
        
        label_name = os.path.splitext(self.imgs[idx])[0] + '.txt'
        label_path = os.path.join(self.lbls_dir, label_name)

        boxes = []
        labels = []

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    # Giả sử format DOTA (8 tọa độ đầu tiên)
                    if len(parts) >= 8:
                        coords = list(map(float, parts[:8]))
                        xs, ys = coords[0::2], coords[1::2]
                        xmin, xmax = min(xs), max(xs)
                        ymin, ymax = min(ys), max(ys)
                        if xmax > xmin and ymax > ymin:
                            boxes.append([xmin, ymin, xmax, ymax])
                            labels.append(1) # Class 1: Ship

        if len(boxes) == 0: # Xử lý ảnh không có tàu
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        target["image_id"] = torch.tensor([idx])
        target["area"] = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        target["iscrowd"] = torch.zeros((len(boxes),), dtype=torch.int64)

        if self.transforms is not None:
            img = self.transforms(img)

        return img, target

    def __len__(self):
        return len(self.imgs)

# 3. KHỞI TẠO DATA LOADER
print(f"--> Đang load dữ liệu từ: {VALIDATION_DIR}")
if os.path.exists(VALIDATION_DIR):
    dataset_val = ShipDataset(VALIDATION_DIR, get_transform(train=False))
    data_loader_val = torch.utils.data.DataLoader(
        dataset_val, 
        batch_size=4,  # Bạn có thể chỉnh batch size tùy ý
        shuffle=False, 
        num_workers=2, 
        collate_fn=collate_fn
    )
    print(f"✅ Đã khởi tạo xong data_loader_val với {len(dataset_val)} ảnh.")
else:
    print(f"❌ LỖI: Không tìm thấy thư mục dữ liệu tại {VALIDATION_DIR}")

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"--> Đang sử dụng thiết bị: {device}")

# 3. KHỞI TẠO MODEL & LOAD WEIGHTS
print("--> Đang khởi tạo kiến trúc model...")
# model = get_model_instance_segmentation(num_classes=2)

# # Đường dẫn file model của bạn
# MODEL_PATH = '/kaggle/input/16122025/faster_rcnn_ship_best(1).pth'

# if os.path.exists(MODEL_PATH):
#     print(f"--> Đang load weights từ: {MODEL_PATH}")
#     try:
#         checkpoint = torch.load(MODEL_PATH, map_location=device)
        
#         # Kiểm tra xem file pth lưu dạng dict hay chỉ state_dict
#         if 'model_state_dict' in checkpoint:
#             model.load_state_dict(checkpoint['model_state_dict'])
#             print("   (Load từ checkpoint dictionary)")
#         else:
#             model.load_state_dict(checkpoint)
#             print("   (Load trực tiếp state_dict)")
            
#         print("✅ ĐÃ LOAD MODEL THÀNH CÔNG!")
#     except Exception as e:
#         print(f"❌ Lỗi khi load model: {e}")
# else:
#     print(f"❌ KHÔNG TÌM THẤY FILE: {MODEL_PATH}")
#     print("Vui lòng kiểm tra lại đường dẫn.")

# # 4. CHUẨN BỊ ĐỂ SỬ DỤNG
# model.to(device)
# model.eval() # Chuyển sang chế độ đánh giá (không train)
# print("--> Model đã sẵn sàng.")


In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm
import os
import json
import shutil # <--- Bổ sung thư viện này để copy file

# =========================================================
# 1. CẤU HÌNH HỆ THỐNG & CHECKPOINT
# =========================================================
DRIVE_ROOT_DIR = '/kaggle/input/processed-data/processed_mmrotate_data' 
WORKING_DIR = '/kaggle/working'

# --- FILE PATH ---
BEST_MODEL_PATH = os.path.join(WORKING_DIR, 'faster_rcnn_ship_best.pth')
CHECKPOINT_PATH = os.path.join(WORKING_DIR, 'checkpoint.pth')            
HISTORY_PATH = os.path.join(WORKING_DIR, 'history.json')

# --- CẤU HÌNH TRAIN ---
RESUME = False              # 🔥 TRUE: Tìm checkpoint để train tiếp | FALSE: Train lại từ đầu
USE_SMALL_DATASET = False   # True: Chạy test nhỏ (cần copy data) | False: Chạy Full
NUM_FILES_TO_COPY = 50     # Số lượng ảnh copy nếu chạy mode nhỏ

NUM_EPOCHS = 30          
LEARNING_RATE = 1e-4       
WEIGHT_DECAY = 1e-4 
BATCH_SIZE = 4            

# =========================================================
# 2. CHUẨN BỊ DỮ LIỆU (ĐÃ SỬA LỖI COPY DATA)
# =========================================================

# --- Hàm copy data (Lấy từ code cũ của bạn) ---
def smart_copy_subset(src_root, dst_root, subset_name, limit=None):
    src_img = os.path.join(src_root, subset_name, 'images')
    src_lbl = os.path.join(src_root, subset_name, 'labelTxt')
    dst_img = os.path.join(dst_root, subset_name, 'images')
    dst_lbl = os.path.join(dst_root, subset_name, 'labelTxt')
    
    os.makedirs(dst_img, exist_ok=True)
    os.makedirs(dst_lbl, exist_ok=True)

    if not os.path.exists(src_img): return

    files = sorted(os.listdir(src_img))[:limit]
    # Chỉ in log 1 lần cho gọn
    if len(files) > 0:
        print(f"--> Copy {len(files)} ảnh '{subset_name}' ra bộ nhớ tạm...")
        
    for f in files:
        shutil.copy(os.path.join(src_img, f), os.path.join(dst_img, f))
        lbl = os.path.splitext(f)[0] + '.txt'
        if os.path.exists(os.path.join(src_lbl, lbl)):
            shutil.copy(os.path.join(src_lbl, lbl), os.path.join(dst_lbl, lbl))

# --- Logic quyết định đường dẫn ---
if USE_SMALL_DATASET:
    print(f"⚠️ CHẾ ĐỘ TEST NHỎ: Đang tạo dữ liệu mẫu ({NUM_FILES_TO_COPY} ảnh)...")
    LOCAL_DIR = os.path.join(WORKING_DIR, 'temp_subset_data')
    
    # Xóa cũ đi tạo lại cho sạch
    if os.path.exists(LOCAL_DIR): 
        shutil.rmtree(LOCAL_DIR)
        
    smart_copy_subset(DRIVE_ROOT_DIR, LOCAL_DIR, 'train', limit=NUM_FILES_TO_COPY)
    smart_copy_subset(DRIVE_ROOT_DIR, LOCAL_DIR, 'val', limit=NUM_FILES_TO_COPY)
    
    ROOT_DIR = LOCAL_DIR
else:
    print("🚀 CHẾ ĐỘ FULL DATA: Đọc trực tiếp từ Input.")
    ROOT_DIR = DRIVE_ROOT_DIR

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"--> Thiết bị: {device}")
print(f"--> Dữ liệu tại: {ROOT_DIR}")

# Dataset & DataLoader
# Lưu ý: Đảm bảo get_transform và collate_fn đã được chạy ở cell trước
dataset_train = ShipDataset(os.path.join(ROOT_DIR, 'train'), get_transform(train=True))
dataset_val = ShipDataset(os.path.join(ROOT_DIR, 'val'), get_transform(train=False))

data_loader = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, collate_fn=collate_fn)
data_loader_val = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, collate_fn=collate_fn)

# =========================================================
# 3. KHỞI TẠO MODEL, OPTIMIZER
# =========================================================
model = get_model_instance_segmentation(num_classes=2)
model.to(device)
# if torch.cuda.device_count() > 1:
#     print(f"🔥 Đang kích hoạt chế độ chạy song song trên {torch.cuda.device_count()} GPUs!")
#     # DataParallel sẽ tự động chia Batch Size ra cho các GPU
#     # Ví dụ: Batch=8 --> Mỗi GPU gánh 4 ảnh
#     model = torch.nn.DataParallel(model)


metric = MeanAveragePrecision(iou_type="bbox")
params = [p for p in model.parameters() if p.requires_grad]

# Optimizer & Scheduler
optimizer = optim.AdamW(params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
lr_scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

# Các biến mặc định
start_epoch = 0
best_map = 0.0
history = {
    'train_loss': [], 
    'val_loss': [], 
    'map_50': [],
    'config': {'batch_size': BATCH_SIZE, 'lr': LEARNING_RATE}
}

# =========================================================
# 4. LOGIC LOAD CHECKPOINT
# =========================================================
if RESUME and os.path.exists(CHECKPOINT_PATH):
    print(f"🔄 TÌM THẤY CHECKPOINT: {CHECKPOINT_PATH}")
    try:
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        lr_scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch'] 
        best_map = checkpoint['best_map']
        history = checkpoint['history']
        print(f"✅ Đã khôi phục thành công! Sẽ bắt đầu từ Epoch {start_epoch + 1}")
    except Exception as e:
        print(f"❌ LỖI LOAD CHECKPOINT: {e}")
        print("--> Sẽ train lại từ đầu.")
else:
    print("✨ KHỞI TẠO TRAIN MỚI (Từ đầu)")

# =========================================================
# 5. VÒNG LẶP TRAINING
# =========================================================
print(f"\n🚀 BẮT ĐẦU HUẤN LUYỆN TỪ EPOCH {start_epoch + 1} ĐẾN {NUM_EPOCHS}...")

for epoch in range(start_epoch, NUM_EPOCHS):
    # --- A. TRAIN ---
    model.train()
    epoch_loss = 0
    progress_bar = tqdm(data_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]", leave=False)
    
    for images, targets in progress_bar:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()
        progress_bar.set_postfix({'loss': f"{losses.item():.4f}"})
    
    lr_scheduler.step()
    avg_train_loss = epoch_loss / len(data_loader)

    # --- B. VALIDATION ---
    val_loss_accum = 0
    with torch.no_grad():
        for images, targets in data_loader_val:
            images = list(image.to(device) for image in images)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets) 
            val_loss_accum += sum(loss for loss in loss_dict.values()).item()
    avg_val_loss = val_loss_accum / len(data_loader_val)

    # Tính mAP
    model.eval() 
    metric.reset()
    with torch.no_grad():
        for images, targets in tqdm(data_loader_val, desc="[Val]", leave=False):
            images = list(image.to(device) for image in images)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            preds = model(images)
            metric.update(preds, targets)
    
    val_stats = metric.compute()
    map_50 = val_stats['map_50'].item()

    # --- C. CẬP NHẬT LỊCH SỬ ---
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['map_50'].append(map_50)
    
    print(f"✅ EPOCH {epoch+1}: Train Loss={avg_train_loss:.4f} | Test Loss={avg_val_loss:.4f} | mAP={map_50:.4f}")

    # =========================================================
    # 6. LƯU CHECKPOINT
    # =========================================================
    checkpoint_data = {
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': lr_scheduler.state_dict(),
        'best_map': best_map,
        'history': history
    }
    
    torch.save(checkpoint_data, CHECKPOINT_PATH)
    with open(HISTORY_PATH, 'w') as f:
        json.dump(history, f)

    if map_50 > best_map:
        best_map = map_50
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"🔥 KỶ LỤC MỚI! Đã lưu Best Model (mAP: {map_50:.4f})")

print(f"\n🏆 HOÀN TẤT! Checkpoint saved: {CHECKPOINT_PATH}")

In [ ]:
import json
import torch
import torchvision.ops as ops
import numpy as np
from tqdm.auto import tqdm
import os

# --- CẤU HÌNH ---
IOU_THRESHOLD = 0.5
CONF_THRESHOLD = 0.05  # Lọc bớt box rác điểm thấp để tính nhanh hơn
SAVE_DIR = '/kaggle/working'

# Tên các file sẽ lưu
FILE_CM = os.path.join(SAVE_DIR, 'faster_rcnn_cm.json')
FILE_PR = os.path.join(SAVE_DIR, 'faster_rcnn_pr.json')
# File history.json đã được lưu trong quá trình train, đảm bảo biến history còn giữ giá trị
FILE_HISTORY = os.path.join(SAVE_DIR, 'history.json')

print("🚀 ĐANG CHẠY ĐÁNH GIÁ ĐỂ LƯU DỮ LIỆU VẼ BIỂU ĐỒ...")

# Đảm bảo model đang ở chế độ eval và dùng device đúng
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
model.eval()

# Biến lưu trữ
detections = []       # Dùng cho PR Curve
tp_count = 0          # Dùng cho Confusion Matrix
fp_count = 0
fn_count = 0
total_gt = 0

with torch.no_grad():
    # Duyệt qua tập Val (hoặc Test)
    for images, targets in tqdm(data_loader_val, desc="Evaluating"):
        images = list(image.to(device) for image in images)
        outputs = model(images)

        for i, output in enumerate(outputs):
            pred_boxes = output['boxes'].cpu()
            pred_scores = output['scores'].cpu()
            gt_boxes = targets[i]['boxes'].cpu()
            
            total_gt += len(gt_boxes)

            # Lọc ngưỡng Confidence thấp
            keep = pred_scores > CONF_THRESHOLD
            pred_boxes = pred_boxes[keep]
            pred_scores = pred_scores[keep]

            # --- XỬ LÝ CHO CONFUSION MATRIX & PR CURVE ---
            if len(pred_boxes) == 0:
                fn_count += len(gt_boxes)
                continue

            if len(gt_boxes) == 0:
                fp_count += len(pred_boxes)
                for score in pred_scores:
                    detections.append({'score': score.item(), 'is_tp': False})
                continue

            # Tính IoU
            iou_matrix = ops.box_iou(pred_boxes, gt_boxes)
            gt_matched = torch.zeros(len(gt_boxes), dtype=torch.bool)

            # Sắp xếp box dự đoán theo điểm giảm dần
            sorted_indices = torch.argsort(pred_scores, descending=True)

            for idx in sorted_indices:
                score = pred_scores[idx].item()
                if iou_matrix.shape[1] > 0:
                    max_iou, max_gt_idx = torch.max(iou_matrix[idx], dim=0)
                else:
                    max_iou = 0

                is_tp = False
                if max_iou >= IOU_THRESHOLD:
                    if not gt_matched[max_gt_idx]:
                        is_tp = True
                        gt_matched[max_gt_idx] = True
                    else:
                        is_tp = False # Trùng lặp (Duplicate)
                
                # Lưu cho PR Curve
                detections.append({'score': score, 'is_tp': is_tp})

                # Cộng dồn cho Confusion Matrix
                if is_tp:
                    tp_count += 1
                else:
                    fp_count += 1
            
            # Số tàu thật chưa được match là FN
            fn_count += (len(gt_boxes) - gt_matched.sum().item())

# --- 1. TÍNH TOÁN DỮ LIỆU PR CURVE ---
detections.sort(key=lambda x: x['score'], reverse=True)
tps = np.array([1 if d['is_tp'] else 0 for d in detections])
fps = np.array([1 if not d['is_tp'] else 0 for d in detections])

tp_cumsum = np.cumsum(tps)
fp_cumsum = np.cumsum(fps)

precisions = tp_cumsum / (tp_cumsum + fp_cumsum + 1e-6)
recalls = tp_cumsum / (total_gt + 1e-6)
ap = np.trapz(precisions, recalls)

# --- 2. LƯU FILE JSON ---

# A. Lưu dữ liệu Confusion Matrix
cm_data = {
    "model": "Faster R-CNN",
    "TP": int(tp_count),
    "FP": int(fp_count),
    "FN": int(fn_count),
    "IOU_THRESHOLD": IOU_THRESHOLD
}
with open(FILE_CM, 'w') as f:
    json.dump(cm_data, f, indent=4)

# B. Lưu dữ liệu PR Curve (Lấy mẫu 1000 điểm để file nhẹ nếu quá dài)
step = max(1, len(precisions) // 1000)
pr_data = {
    "model": "Faster R-CNN",
    "AP": float(ap),
    "precisions": precisions[::step].tolist(), # Chuyển numpy -> list để lưu json
    "recalls": recalls[::step].tolist()
}
with open(FILE_PR, 'w') as f:
    json.dump(pr_data, f) # Không indent để file nhỏ gọn

# C. Lưu lại History (nếu chưa lưu)
if 'history' in globals():
    with open(FILE_HISTORY, 'w') as f:
        json.dump(history, f, indent=4)

print(f"✅ ĐÃ LƯU XONG!")
print(f"   - CM Data: {FILE_CM}")
print(f"   - PR Data: {FILE_PR}")
print(f"   - History: {FILE_HISTORY}")
print(f"👉 Hãy tải 3 file này về và dùng code vẽ biểu đồ tương ứng.")

In [ ]:
# Vẽ
import json
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import os

# ==============================================================================
# 1. CẤU HÌNH GIAO DIỆN OCEANEYE
# ==============================================================================
BG_COLOR = '#0a192f'     # Nền xanh đen
TEXT_COLOR = '#ccd6f6'   # Chữ trắng xám
ACC_COLOR = '#64ffda'    # Màu Cyan (cho mAP/Precision)
LOSS_COLOR = '#ff6b6b'   # Màu Đỏ cam (cho Loss)
GRID_COLOR = '#112240'   # Màu lưới mờ
CM_COLOR = 'Blues'       # Màu cho Confusion Matrix

plt.rcParams.update({
    'text.color': TEXT_COLOR,
    'axes.labelcolor': TEXT_COLOR,
    'xtick.color': TEXT_COLOR,
    'ytick.color': TEXT_COLOR,
    'axes.facecolor': BG_COLOR,
    'figure.facecolor': BG_COLOR,
    'grid.color': GRID_COLOR,
    'font.size': 12
})

# Đường dẫn file dữ liệu (Bạn đã tạo ở bước trước)
DIR = '/kaggle/working'
FILE_HISTORY = os.path.join(DIR, 'history.json')
FILE_CM = os.path.join(DIR, 'faster_rcnn_cm.json')
FILE_PR = os.path.join(DIR, 'faster_rcnn_pr.json')

# ==============================================================================
# 2. HÀM VẼ BIỂU ĐỒ TRAINING HISTORY
# ==============================================================================
def draw_training_history():
    if not os.path.exists(FILE_HISTORY):
        print(f"⚠️ Không tìm thấy {FILE_HISTORY}, bỏ qua vẽ History.")
        return

    with open(FILE_HISTORY, 'r') as f:
        data = json.load(f)

    epochs = range(1, len(data['train_loss']) + 1)
    train_loss = data['train_loss']
    val_map = data['map_50']

    fig, ax1 = plt.subplots(figsize=(12, 6))

    # Trục 1: Loss
    ax1.set_xlabel('Epochs', fontweight='bold')
    ax1.set_ylabel('Loss', color=LOSS_COLOR, fontweight='bold')
    line1 = ax1.plot(epochs, train_loss, color=LOSS_COLOR, marker='o', label='Train Loss', linewidth=2)
    ax1.tick_params(axis='y', labelcolor=LOSS_COLOR)
    ax1.grid(True, linestyle='--', alpha=0.5)

    # Trục 2: mAP
    ax2 = ax1.twinx()
    ax2.set_ylabel('mAP@50', color=ACC_COLOR, fontweight='bold')
    line2 = ax2.plot(epochs, val_map, color=ACC_COLOR, marker='s', label='mAP@50', linewidth=2)
    ax2.tick_params(axis='y', labelcolor=ACC_COLOR)

    # Tiêu đề & Chú thích
    plt.title('FASTER R-CNN TRAINING PERFORMANCE', fontsize=16, fontweight='bold', pad=20)
    
    # Gộp legend
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='center right', facecolor=BG_COLOR, edgecolor=TEXT_COLOR)

    # Lưu ảnh
    save_path = os.path.join(DIR, 'chart_history.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor=BG_COLOR)
    print(f"✅ Đã vẽ xong History: {save_path}")
    plt.show()

# ==============================================================================
# 3. HÀM VẼ CONFUSION MATRIX
# ==============================================================================
def draw_confusion_matrix():
    if not os.path.exists(FILE_CM):
        print(f"⚠️ Không tìm thấy {FILE_CM}, bỏ qua vẽ CM.")
        return

    with open(FILE_CM, 'r') as f:
        data = json.load(f)

    tp = data['TP']
    fp = data['FP']
    fn = data['FN']
    tn = 0 # Object Detection thường không tính True Negative nền

    # Tạo ma trận 2x2
    cm = np.array([[tp, fn], [fp, tn]])
    
    # Tính phần trăm để hiển thị
    cm_sum = np.sum(cm)
    cm_perc = cm / cm_sum if cm_sum > 0 else cm

    labels = np.array([
        [f"{tp}\n(TP - Đúng)", f"{fn}\n(FN - Sót)"],
        [f"{fp}\n(FP - Báo giả)", f"N/A\n(TN - Nền)"]
    ])

    fig, ax = plt.subplots(figsize=(8, 6))
    
    # Vẽ Heatmap
    sns.heatmap(cm, annot=labels, fmt='', cmap=CM_COLOR, cbar=False, 
                linewidths=1, linecolor=GRID_COLOR,
                xticklabels=['Predicted Ship', 'Predicted Background'],
                yticklabels=['Actual Ship', 'Actual Background'],
                annot_kws={"size": 12, "weight": "bold"})

    plt.title(f"CONFUSION MATRIX (IoU={data.get('IOU_THRESHOLD', 0.5)})", 
              fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Prediction', fontsize=12, fontweight='bold')
    plt.ylabel('Ground Truth', fontsize=12, fontweight='bold')

    # Chỉnh màu chữ trong ô
    for text in ax.texts:
        if 'TP' in text.get_text() or 'FN' in text.get_text():
            text.set_color('black') # Màu chữ cho dễ đọc trên nền nhạt/đậm

    save_path = os.path.join(DIR, 'chart_confusion_matrix.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor=BG_COLOR)
    print(f"✅ Đã vẽ xong Confusion Matrix: {save_path}")
    plt.show()

# ==============================================================================
# 4. HÀM VẼ PRECISION-RECALL CURVE
# ==============================================================================
def draw_pr_curve():
    if not os.path.exists(FILE_PR):
        print(f"⚠️ Không tìm thấy {FILE_PR}, bỏ qua vẽ PR Curve.")
        return

    with open(FILE_PR, 'r') as f:
        data = json.load(f)

    precisions = data['precisions']
    recalls = data['recalls']
    ap = data['AP']

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.plot(recalls, precisions, color=ACC_COLOR, linewidth=3, label=f'Faster R-CNN (AP={ap:.2f})')
    ax.fill_between(recalls, precisions, color=ACC_COLOR, alpha=0.2)

    ax.set_title('PRECISION-RECALL CURVE', fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('Recall (Độ phủ)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Precision (Độ chính xác)', fontsize=12, fontweight='bold')
    
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.grid(True, linestyle='--', color=GRID_COLOR, alpha=0.5)
    
    # Legend
    ax.legend(loc='lower left', facecolor=BG_COLOR, edgecolor=TEXT_COLOR)

    # Viền trục
    ax.spines['bottom'].set_color(TEXT_COLOR)
    ax.spines['left'].set_color(TEXT_COLOR)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    save_path = os.path.join(DIR, 'chart_pr_curve.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor=BG_COLOR)
    print(f"✅ Đã vẽ xong PR Curve: {save_path}")
    plt.show()

# ==============================================================================
# CHẠY TẤT CẢ
# ==============================================================================
print("🚀 ĐANG VẼ BIỂU ĐỒ...")
draw_training_history()
draw_confusion_matrix()
draw_pr_curve()

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
import torch
import torchvision.ops as ops
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import random

# =========================================================
# CẤU HÌNH TEST
# =========================================================
SCORE_THRESHOLD = 0.6  # Chỉ lấy box có độ tin cậy > 60%
IOU_THRESHOLD = 0.5    # Ngưỡng chồng lấn để tính là "Đúng" (TP)

ROOT_DIR = '/kaggle/input/processed-data/processed_mmrotate_data/test'
# Load Dataset Test
# (Nếu bạn không có folder 'test', code sẽ tự động dùng folder 'val' để demo)
test_root = os.path.join(ROOT_DIR, 'test')
if not os.path.exists(os.path.join(test_root, 'images')):
    print("⚠️ Không tìm thấy tập Test riêng, sử dụng tập Validation để đánh giá.")
    test_root = os.path.join(ROOT_DIR, 'val')

ds_test = ShipDataset(root=test_root, transforms=get_transform(train=False))

# =========================================================
# PHẦN 1: VISUALIZE SO SÁNH (GROUND TRUTH vs PREDICTION)
# =========================================================
def visualize_comparisons(model, dataset, num_samples=6):
    model.eval()
    print(f"\n📸 --- HIỂN THỊ {num_samples} ẢNH NGẪU NHIÊN ĐỂ SO SÁNH ---")
    print("🟦 Hộp Xanh Dương (Blue):  Ground Truth (Đáp án thật)")
    print("🟩 Hộp Xanh Lá (Green):    Prediction (Model dự đoán)")
    
    # Lấy ngẫu nhiên index
    indices = random.sample(range(len(dataset)), min(num_samples, len(dataset)))
    
    # Setup vẽ nhiều hình
    cols = 2
    rows = (len(indices) + 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, 6 * rows))
    axes = axes.flatten()
    
    for i, idx in enumerate(indices):
        img, target = dataset[idx]
        
        # 1. Dự đoán
        with torch.no_grad():
            prediction = model([img.to(device)])[0]

        # Chuẩn bị ảnh nền
        img_np = img.mul(255).permute(1, 2, 0).byte().cpu().numpy().copy()
        img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR) # Chuyển sang BGR để vẽ bằng OpenCV
        
        # 2. Vẽ Ground Truth (Màu Xanh Dương: 255, 0, 0 trong BGR)
        gt_boxes = target['boxes'].cpu().numpy()
        for box in gt_boxes:
            cv2.rectangle(img_np, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (255, 0, 0), 2)
            
        # 3. Vẽ Prediction (Màu Xanh Lá: 0, 255, 0)
        pred_boxes = prediction['boxes']
        pred_scores = prediction['scores']
        
        # Lọc Score
        keep = pred_scores > SCORE_THRESHOLD
        pred_boxes = pred_boxes[keep]
        pred_scores = pred_scores[keep]
        
        # Lọc NMS (Xóa trùng lặp)
        nms_keep = ops.nms(pred_boxes, pred_scores, 0.3)
        pred_boxes = pred_boxes[nms_keep].cpu().numpy()
        pred_scores = pred_scores[nms_keep].cpu().numpy()
        
        for j, box in enumerate(pred_boxes):
            cv2.rectangle(img_np, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (0, 255, 0), 2)
            # Ghi điểm số
            cv2.putText(img_np, f"{pred_scores[j]:.2f}", (int(box[0]), int(box[1])-5), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        # Hiển thị lên Matplotlib
        img_rgb = cv2.cvtColor(img_np, cv2.COLOR_BGR2RGB)
        axes[i].imshow(img_rgb)
        axes[i].set_title(f"Image {idx} | GT: {len(gt_boxes)} | Pred: {len(pred_boxes)}")
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

# --- CHẠY ---
if 'model' in globals():
    visualize_comparisons(model, ds_test, num_samples=6)
    evaluate_statistics(model, ds_test, score_thresh=SCORE_THRESHOLD)
else:
    print("❌ Chưa có model. Hãy chạy cell training trước!")

In [ ]:
import gradio as gr
import torch
import torchvision.transforms as T
import cv2
import numpy as np
import torchvision.ops as ops
import os

# ==========================================
# 1. CẤU HÌNH & LOAD MODEL
# ==========================================
# Đường dẫn model tốt nhất (đảm bảo bạn đã train xong)
MODEL_PATH = '/kaggle/working/faster_rcnn_ship_best.pth'
DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# Cấu hình bộ lọc (Chỉnh ở đây nếu muốn kết quả khác)
SCORE_THRESH = 0.6   # Độ tin cậy tối thiểu
IOU_THRESH = 0.3     # Ngưỡng lọc trùng lặp NMS

# Khởi tạo model (nếu chưa có trong bộ nhớ)
try:
    model
except NameError:
    # Nếu biến model chưa tồn tại thì khởi tạo lại
    model = get_model_instance_segmentation(num_classes=2)

model.to(DEVICE)


# Load weights
if os.path.exists(MODEL_PATH):
    print(f"--> Đang load model từ: {MODEL_PATH}")
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
else:
    print("⚠️ CẢNH BÁO: Không tìm thấy file model! Đang dùng model ngẫu nhiên (sẽ dự đoán sai).")

model.eval()

# ==========================================
# 2. HÀM XỬ LÝ ẢNH & DỰ ĐOÁN
# ==========================================
def predict_image(input_image):
    if input_image is None:
        return None

    # Gradio trả về ảnh dạng Numpy (H, W, C), cần chuyển sang Tensor
    # input_image đang là RGB
    transform = T.Compose([T.ToTensor()])
    img_tensor = transform(input_image).to(DEVICE)

    with torch.no_grad():
        prediction = model([img_tensor])[0]

    # Lấy kết quả
    boxes = prediction['boxes']
    scores = prediction['scores']

    # --- LỌC KẾT QUẢ ---
    # 1. Lọc theo điểm số (Score Threshold)
    keep_score = scores > SCORE_THRESH
    boxes = boxes[keep_score]
    scores = scores[keep_score]

    # 2. Lọc chồng lấn (NMS)
    keep_nms = ops.nms(boxes, scores, IOU_THRESH)
    final_boxes = boxes[keep_nms].cpu().numpy()
    final_scores = scores[keep_nms].cpu().numpy()

    # --- VẼ HÌNH ---
    # Chuyển ảnh từ RGB sang BGR để vẽ bằng OpenCV (vì OpenCV dùng BGR)
    # Lưu ý: Gradio input là RGB, nhưng OpenCV vẽ lên cần BGR, sau đó lại convert về RGB để hiển thị
    img_draw = cv2.cvtColor(input_image, cv2.COLOR_RGB2BGR)

    for i, box in enumerate(final_boxes):
        x1, y1, x2, y2 = map(int, box)
        score = final_scores[i]

        # Vẽ hộp chữ nhật màu xanh lá
        cv2.rectangle(img_draw, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        # Viết điểm số
        label = f"Ship: {score:.2f}"
        cv2.putText(img_draw, label, (x1, y1 - 10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Chuyển lại về RGB để hiển thị trên Web
    return cv2.cvtColor(img_draw, cv2.COLOR_BGR2RGB)

# ==========================================
# 3. TẠO GIAO DIỆN WEB
# ==========================================
iface = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="numpy", label="Upload Ảnh Vệ Tinh/Biển"),
    outputs=gr.Image(type="numpy", label="Kết Quả Phát Hiện"),
    title="🚢 Ship Detection App",
    description="Upload ảnh vệ tinh để model phát hiện tàu bè. (Model: Faster R-CNN)",
    examples=None # Bạn có thể thêm list đường dẫn ảnh mẫu vào đây nếu muốn
)

# share=True để tạo link công khai (chạy được trên Kaggle)
print("--> Đang khởi động Web App...")
iface.launch(share=True, debug=True)

In [ ]:
!pip install -q gradio

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import json
import os
import cv2
from tqdm.auto import tqdm
import torchvision.ops as ops

# =========================================================
# 1. CẤU HÌNH & CHUẨN BỊ
# =========================================================


WORKING_DIR = '/kaggle/working'
IOU_THRESHOLD = 0.5
CONF_THRESHOLD = 0.05  # Lọc bớt các box rác điểm quá thấp để tính toán nhanh hơn
OUTPUT_IMG_FILE = os.path.join(WORKING_DIR, "faster_rcnn_pr_curve.png")
OUTPUT_JSON_FILE = os.path.join(WORKING_DIR, "faster_rcnn_metrics.json")

print(f"🚀 BẮT ĐẦU ĐÁNH GIÁ (IOU Thresh: {IOU_THRESHOLD})...")

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
model.eval()

# Danh sách chứa toàn bộ kết quả
# Format: {'score': float, 'is_tp': bool}
detections = []
total_ground_truths = 0

# =========================================================
# 2. VÒNG LẶP ĐÁNH GIÁ (INFERENCE LOOP)
# =========================================================
# Sử dụng data_loader_val (hoặc test nếu bạn đã tạo)
eval_loader = data_loader_val

with torch.no_grad():
    for images, targets in tqdm(eval_loader, desc="Evaluating"):
        images = list(image.to(device) for image in images)
        
        # Lấy dự đoán từ model
        outputs = model(images)

        # Xử lý từng ảnh trong batch
        for i, output in enumerate(outputs):
            pred_boxes = output['boxes'].cpu()
            pred_scores = output['scores'].cpu()
            gt_boxes = targets[i]['boxes'].cpu() # Ground Truth boxes
            
            # Đếm tổng số lượng tàu thực tế
            total_ground_truths += len(gt_boxes)

            # Lọc sơ bộ các box điểm thấp để giảm tải tính toán
            keep_idxs = pred_scores > CONF_THRESHOLD
            pred_boxes = pred_boxes[keep_idxs]
            pred_scores = pred_scores[keep_idxs]

            if len(pred_boxes) == 0:
                continue

            if len(gt_boxes) == 0:
                # Nếu ảnh không có tàu mà model dự đoán ra -> Toàn bộ là False Positive
                for score in pred_scores:
                    detections.append({'score': score.item(), 'is_tp': False})
                continue

            # --- TÍNH IOU (Intersection over Union) ---
            # Dùng hàm tối ưu của PyTorch thay vì Shapely (vì Faster RCNN dùng box ngang)
            iou_matrix = ops.box_iou(pred_boxes, gt_boxes)  # shape: [N_pred, M_gt]

            # Đánh dấu GT nào đã được "ăn" (matched)
            gt_matched = torch.zeros(len(gt_boxes), dtype=torch.bool)

            # Sắp xếp các box dự đoán theo điểm giảm dần trong nội bộ ảnh này
            # (Thực ra ta sẽ sort toàn bộ dataset sau, nhưng xử lý cục bộ trước cũng tốt)
            sorted_indices = torch.argsort(pred_scores, descending=True)
            
            for pred_idx in sorted_indices:
                score = pred_scores[pred_idx].item()
                max_iou, max_gt_idx = torch.max(iou_matrix[pred_idx], dim=0)

                is_tp = False
                # Logic: IoU đủ lớn VÀ Ground Truth đó chưa bị box nào khác "ăn"
                if max_iou >= IOU_THRESHOLD:
                    if not gt_matched[max_gt_idx]:
                        is_tp = True
                        gt_matched[max_gt_idx] = True # Đánh dấu đã match
                    else:
                        is_tp = False # Duplicate detection (trùng lặp)
                
                detections.append({'score': score, 'is_tp': is_tp})

# =========================================================
# 3. TÍNH TOÁN PRECISION - RECALL & AP
# =========================================================
# Sắp xếp toàn bộ dự đoán theo Score giảm dần
detections.sort(key=lambda x: x['score'], reverse=True)

tps = np.array([1 if d['is_tp'] else 0 for d in detections])
fps = np.array([1 if not d['is_tp'] else 0 for d in detections])

tp_cumsum = np.cumsum(tps)
fp_cumsum = np.cumsum(fps)

# Tính Precision và Recall
# +1e-6 để tránh chia cho 0
precisions = tp_cumsum / (tp_cumsum + fp_cumsum + 1e-6)
recalls = tp_cumsum / (total_ground_truths + 1e-6)

# Tính AP (Average Precision) bằng quy tắc hình thang (Trapezoidal rule)
ap = np.trapz(precisions, recalls)

print(f"\n📊 KẾT QUẢ THỰC TẾ (Faster R-CNN):")
print(f"   - Tổng số tàu thật (Ground Truth): {total_ground_truths}")
print(f"   - Tổng số box dự đoán: {len(detections)}")
print(f"   - Average Precision (AP @ IoU {IOU_THRESHOLD}): {ap:.4f}")

# =========================================================
# 4. LƯU SỐ LIỆU JSON
# =========================================================
metrics_data = {
    "model": "Faster R-CNN (ResNet50)",
    "mAP": round(ap * 100, 2),
    "AP_IOU_0_50": round(ap * 100, 2),
    "total_gt": int(total_ground_truths),
    "total_preds": len(detections)
}

with open(OUTPUT_JSON_FILE, 'w') as f:
    json.dump(metrics_data, f, indent=4)
print(f"✅ Đã lưu file số liệu: {OUTPUT_JSON_FILE}")

# =========================================================
# 5. VẼ BIỂU ĐỒ (STYLE DARK MODE)
# =========================================================
BG_COLOR = '#0a192f'
TEXT_COLOR = '#ccd6f6'
LINE_COLOR = '#64ffda'

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(BG_COLOR)
ax.set_facecolor(BG_COLOR)

# Vẽ đường cong
ax.plot(recalls, precisions, color=LINE_COLOR, linewidth=3, label=f'Faster R-CNN (AP={ap:.2f})')
ax.fill_between(recalls, precisions, color=LINE_COLOR, alpha=0.15)

# Trang trí
ax.set_title(f'PRECISION-RECALL CURVE (IoU={IOU_THRESHOLD})', fontsize=16, fontweight='bold', color=TEXT_COLOR, pad=20)
ax.set_xlabel('Recall (Độ phủ)', fontsize=12, fontweight='bold', color=TEXT_COLOR)
ax.set_ylabel('Precision (Độ chính xác)', fontsize=12, fontweight='bold', color=TEXT_COLOR)

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])

ax.grid(True, linestyle='--', color='#112240', alpha=0.5)
ax.spines['bottom'].set_color(TEXT_COLOR)
ax.spines['left'].set_color(TEXT_COLOR)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(colors=TEXT_COLOR)

leg = ax.legend(loc="lower left", facecolor=BG_COLOR, edgecolor=TEXT_COLOR)
for text in leg.get_texts(): 
    text.set_color(TEXT_COLOR)

plt.tight_layout()
plt.savefig(OUTPUT_IMG_FILE, dpi=300, facecolor=BG_COLOR)
print(f"✅ Đã lưu ảnh biểu đồ: {OUTPUT_IMG_FILE}")
plt.show()